## Modélisation des chiffres d'affaires

### 1. Chargement des données

Chargement des jeux d'entraînement et de test préparés lors du feature engineering.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

DATA_DIR = Path("../data/Train_Test")

X_train = pd.read_csv(DATA_DIR / "X_train.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv")

X_test = pd.read_csv(DATA_DIR / "X_test.csv")
y_test = pd.read_csv(DATA_DIR / "y_test.csv")

print("Dimensions des données :")
print("X_train :", X_train.shape)
print("y_train :", y_train.shape)
print("X_test  :", X_test.shape)
print("y_test  :", y_test.shape)

Dimensions des données :
X_train : (578030, 25)
y_train : (578030, 1)
X_test  : (145032, 25)
y_test  : (145032, 1)


In [ ]:
# controle de la qualité des données

print("Valeurs manquantes :")
print("X_train :", X_train.isna().sum().sum())
print("y_train :", y_train.isna().sum().sum())


print("\nValeurs infinies :")
print("X_train :", np.isinf(X_train.select_dtypes(include=np.number)).sum().sum())

print("\nStatistiques de la cible :")
print(y_train.describe())

Valeurs manquantes :
X_train : 0
y_train : 0

Valeurs infinies :
X_train : 0

Statistiques de la cible :
               Sales
count  578030.000000
mean     6866.063014
std      2323.776407
min      2574.000000
25%      5083.000000
50%      6671.000000
75%      8334.000000
max     21387.000000


### 2. Préparation de la validation temporelle

Les données étant organisées par date et par magasin, les séparations temporelles sont effectuées par journée complète afin d'éviter les mélanges entre entraînement et validation.

In [4]:
from sklearn.model_selection import TimeSeriesSplit

def reconstruire_dates(X):
    return pd.to_datetime({
        "year": X["Year"],
        "month": X["Month"],
        "day": X["Day"]
    })

dates_train = reconstruire_dates(X_train)
dates_test = reconstruire_dates(X_test)

print("Période d'entraînement :")
print(dates_train.min(), "→", dates_train.max())

print("\nPériode de test :")
print(dates_test.min(), "→", dates_test.max())

print("\nNombre de dates d'entraînement :", dates_train.nunique())
print("Nombre de dates de test :", dates_test.nunique())

print("\nEntraînement chronologique :", dates_train.is_monotonic_increasing)
print("Test chronologique :", dates_test.is_monotonic_increasing)
print("Séparation temporelle respectée :", dates_train.max() < dates_test.min())

Période d'entraînement :
2013-01-01 00:00:00 → 2014-07-31 00:00:00

Période de test :
2014-08-01 00:00:00 → 2014-12-23 00:00:00

Nombre de dates d'entraînement : 577
Nombre de dates de test : 145

Entraînement chronologique : True
Test chronologique : True
Séparation temporelle respectée : True


### 3. Création des folds temporels

Création de trois découpages temporels à fenêtre croissante avec `TimeSeriesSplit`.

Les dates sont séparées avant de retrouver les lignes correspondantes afin de conserver tous les magasins d'une même journée dans le même ensemble.

Ces folds seront réutilisés pour entraîner et comparer les modèles sur les mêmes périodes de validation.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

# récupérer les dates uniques dans l'ordre chronologique
dates_uniques = np.sort(dates_train.unique())

# créer trois découpages temporels
tscv = TimeSeriesSplit(n_splits=3)

folds = []

for train_idx, val_idx in tscv.split(dates_uniques):

    dates_fold_train = dates_uniques[train_idx]
    dates_fold_val = dates_uniques[val_idx]

    lignes_train = np.flatnonzero(
        dates_train.isin(dates_fold_train)
    )

    lignes_val = np.flatnonzero(
        dates_train.isin(dates_fold_val)
    )

    folds.append((lignes_train, lignes_val))

print("Nombre de folds créés :", len(folds))

Nombre de folds créés : 3


In [ ]:
# séparer les dates uniques pour garder chaque journée entière
dates_uniques = np.sort(dates_train.unique())

tscv = TimeSeriesSplit(n_splits=3)

folds = []

for train_idx, val_idx in tscv.split(dates_uniques):
    dates_fold_train = dates_uniques[train_idx]
    dates_fold_val = dates_uniques[val_idx]

    # récupérer toutes les lignes correspondant aux dates
    lignes_train = np.flatnonzero(
        dates_train.isin(dates_fold_train)
    )

    lignes_val = np.flatnonzero(
        dates_train.isin(dates_fold_val)
    )

    folds.append((lignes_train, lignes_val))

print("Nombre de folds :", len(folds))

for i, (train_idx, val_idx) in enumerate(folds, start=1):
    dates_fold_train = dates_train.iloc[train_idx]
    dates_fold_val = dates_train.iloc[val_idx]

    print(f"\nFold {i}")
    print("Entraînement :", len(train_idx), "lignes")
    print("Validation  :", len(val_idx), "lignes")
    print(
        "Période entraînement :",
        dates_fold_train.min().date(),
        "→",
        dates_fold_train.max().date()
    )
    print(
        "Période validation :",
        dates_fold_val.min().date(),
        "→",
        dates_fold_val.max().date()
    )
    print(
        "Séparation respectée :",
        dates_fold_train.max() < dates_fold_val.min()
    )

Nombre de folds : 3

Fold 1
Entraînement : 145844 lignes
Validation  : 143927 lignes
Période entraînement : 2013-01-01 → 2013-05-25
Période validation : 2013-05-26 → 2013-10-16
Séparation respectée : True

Fold 2
Entraînement : 289771 lignes
Validation  : 143759 lignes
Période entraînement : 2013-01-01 → 2013-10-16
Période validation : 2013-10-17 → 2014-03-09
Séparation respectée : True

Fold 3
Entraînement : 433530 lignes
Validation  : 144500 lignes
Période entraînement : 2013-01-01 → 2014-03-09
Période validation : 2014-03-10 → 2014-07-31
Séparation respectée : True
